In [2]:
import sqlite3
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils.dataframe import dataframe_to_rows

# --- Paths ---
db_path = r"C:\Users\Lenovo\OneDrive\Desktop\datafrank\norwegian-electricity-analysis\NO2_forcasting\data\sqlite\electricity_prices_no.db"
excel_path = r"C:\Users\Lenovo\OneDrive\Desktop\datafrank\norwegian-electricity-analysis\NO2_forcasting\data\sqlite\tsa.xlsx"

# --- Connect to SQLite ---
conn = sqlite3.connect(db_path)

# --- Load existing Excel file to find the latest timestamp ---
wb = load_workbook(excel_path)
ws = wb.active

# Get the latest timestamp from the existing data (column B, starting row 2)
# Find the last row with data
last_row = ws.max_row
if last_row > 1:
    # Get latest timestamp from existing data (assumes data is sorted DESC)
    latest_timestamp = ws.cell(row=2, column=2).value  # B2 = most recent timestamp
else:
    latest_timestamp = None

# --- Query only NEW records ---
if latest_timestamp:
    query = f"""
    SELECT *
    FROM electricity_prices_historical
    WHERE timestamp > '{latest_timestamp}'
    ORDER BY timestamp DESC;
    """
else:
    # If no existing data, get everything
    query = """
    SELECT *
    FROM electricity_prices_historical
    ORDER BY timestamp DESC;
    """

# --- Load new data into DataFrame ---
df_new = pd.read_sql_query(query, conn)
conn.close()

# --- Check if there's new data to add ---
if df_new.empty:
    print("No new data to add.")
else:
    # --- Insert new rows at the TOP (after header, before existing data) ---
    # Shift existing data down
    num_new_rows = len(df_new)
    
    # Insert blank rows at the top (after header row 1)
    ws.insert_rows(2, num_new_rows)
    
    # Write new data starting at row 2
    for r_idx, row in enumerate(dataframe_to_rows(df_new, index=False, header=False), start=2):
        for c_idx, value in enumerate(row, start=1):
            ws.cell(row=r_idx, column=c_idx, value=value)
    
    # --- Update the Table range ---
    table = ws.tables["PriceData"]
    new_last_row = ws.max_row
    table.ref = f"A1:C{new_last_row}"
    
    # --- Save the workbook ---
    wb.save(excel_path)
    
    print(f"Added {num_new_rows} new rows.")
    print(f"Table 'PriceData' updated to range: A1:C{new_last_row}")
    print(f"Export complete: {excel_path}")

No new data to add.
